# FIUBench Forget5 재평가 — fast tokenizer + 원본 generation 경계

이 노트북은 기존 **GA 또는 PO 최종 `checkpoint.pt`**를 다시 학습하지 않고 평가합니다.

해결하는 문제:

- `tokenizers==0.15.1`이 모델의 `tokenizer.json`을 읽지 못한 문제
- `use_fast=False` 우회 뒤 `labels != -100` 경계 패치가 `<|assistant|>`를 제거한 문제
- 평가 중 checkpoint 경로 혼동과 중간 `step_*` checkpoint 오선택
- Kaggle T4에서 Flash Attention을 사용할 수 없는 문제

평가 환경은 예전 성공 환경과 맞춘 `transformers 4.46.3`, `tokenizers 0.20.3`,
`peft 0.13.2`, `accelerate 0.34.2`입니다. `LlamaTokenizerFast`와 공개 원본의
문자열 answer-tag 분리가 실제로 호환되는지 사전 검증에 통과해야 평가가 시작됩니다.

## 시작 전

1. Kaggle에서 GPU T4를 켭니다.
2. GA 또는 PO 학습 Notebook의 저장된 Version Output을 **Add Input**으로 하나만 연결합니다.
3. 셀을 순서대로 실행합니다.
4. 설치 셀 다음에 커널 재시작 셀을 한 번 실행합니다.

평가는 두 부분으로 나뉩니다.

- **Core**: Forget5/Retain5의 ROUGE, EM, loss, Truth Ratio, Min-K 계열
- **APE**: Forget5 paraphrased question 1,200개. 선택 실행

`GPT-Eval`, `MME`, `POPE`, 완전한 MIA/KS 집계는 이 노트북의 범위가 아닙니다.


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working
if [ ! -d FIUBench_kaggle ]; then
  git clone https://github.com/npngjjm/FIUBench_kaggle.git
fi

python -m pip uninstall -y torchao || true
python -m pip install --no-cache-dir --upgrade \
  'transformers==4.46.3' \
  'tokenizers==0.20.3' \
  'peft==0.13.2' \
  'accelerate==0.34.2' \
  'huggingface_hub==0.36.2' \
  'numpy==2.1.3' \
  hydra-core omegaconf datasets sentencepiece protobuf scipy pillow \
  pyyaml requests tqdm rouge_score scikit-learn

echo 'INSTALL_COMPLETE — 다음 셀에서 Python 커널을 재시작하세요.'


설치가 끝난 뒤 바로 아래 셀을 한 번 실행하세요. Python 프로세스만 재시작하며,
`/kaggle/working`의 clone과 설치된 패키지는 유지됩니다. 연결이 잠시 끊겼다가 자동 복구됩니다.
그 후에는 이 셀을 다시 실행하지 말고 다음 셀부터 진행하세요.


In [ ]:
import os
os._exit(0)


아래 `METHOD`를 연결한 checkpoint에 맞게 `"GA"` 또는 `"PO"`로 지정하세요.


In [ ]:
METHOD = "GA"  # "GA" 또는 "PO"

assert METHOD in {"GA", "PO"}
print("평가 방법:", METHOD)


In [ ]:
import os
import shutil
import sys
from importlib.metadata import version
from pathlib import Path

import torch

repo = Path("/kaggle/working/FIUBench_kaggle")
assert repo.is_dir(), "저장소가 없습니다. 첫 설치/clone 셀을 다시 실행하세요."
os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

expected = {
    "transformers": "4.46.3",
    "tokenizers": "0.20.3",
    "peft": "0.13.2",
    "accelerate": "0.34.2",
    "huggingface-hub": "0.36.2",
    "numpy": "2.1.3",
}

print("=== 라이브러리 ===")
for package, wanted in expected.items():
    installed = version(package)
    print(package, installed)
    assert installed == wanted, (package, installed, wanted)

assert torch.cuda.is_available(), "Kaggle에서 GPU를 켜세요."
print("torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

# Add Input에서 최종 checkpoint만 선택합니다. step_* 중간 checkpoint는 제외합니다.
candidates = [
    p for p in Path("/kaggle/input").rglob("checkpoint.pt")
    if not any(part.startswith("step_") for part in p.parts)
]

print("최종 checkpoint 후보:", *map(str, candidates), sep="\n")
assert len(candidates) == 1, (
    "최종 checkpoint 후보가 정확히 하나여야 합니다. "
    "GA 또는 PO 학습 Output 하나만 Add Input으로 연결하세요."
)

run_name = f"{METHOD.lower()}_forget5_fast_eval"
run_dir = Path("/kaggle/working") / run_name
checkpoint = run_dir / "checkpoint.pt"
run_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(candidates[0], checkpoint)

state = torch.load(checkpoint, map_location="cpu", weights_only=True)
assert isinstance(state, dict) and state
print("checkpoint:", checkpoint)
print("checkpoint 텐서 수:", len(state))
print("checkpoint 크기 MiB:", round(checkpoint.stat().st_size / 1024**2, 1))
del state


In [ ]:
# fast tokenizer가 원본 evaluate_util.py의 answer boundary와 호환되는지 확인합니다.
from transformers import AutoTokenizer
from utils import get_model_identifiers_from_yaml

model_path = "gray311/vlm_unlearning_ft_llava_phi_3_mini"
model_family = "llava-phi"

tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)
assert tokenizer.is_fast

model_cfg = get_model_identifiers_from_yaml(model_family)
question_tag = model_cfg["question_start_tag"]
answer_tag = model_cfg["answer_tag"]
original_boundary = answer_tag.replace("\n", "")

conversation = (
    question_tag
    + "<image>\n"
    + "What is the full name of the person in the image?"
    + answer_tag
    + "The full name is test person."
)
ids = tokenizer(conversation, return_tensors="pt")["input_ids"]
decoded = tokenizer.batch_decode(ids, skip_special_tokens=False)[0]

print("tokenizer class:", tokenizer.__class__.__name__)
print("is_fast:", tokenizer.is_fast)
print("원본 boundary:", repr(original_boundary))
print("decoded:", repr(decoded))
print("boundary 발견:", original_boundary in decoded)

assert tokenizer.__class__.__name__ == "LlamaTokenizerFast"
assert original_boundary in decoded, (
    "Fast tokenizer에서도 공개 원본의 answer boundary를 찾지 못했습니다. "
    "이 상태에서는 평가를 실행하지 마세요."
)

prompt = decoded.split(original_boundary, 1)[0].strip(" ") + original_boundary
prompt_ids = tokenizer(prompt, add_special_tokens=True)["input_ids"]
assistant_id = tokenizer.convert_tokens_to_ids("<|assistant|>")
assert prompt_ids[-1] == assistant_id, (
    "생성 prompt가 <|assistant|> 토큰으로 끝나지 않습니다."
)
print("생성 prompt 마지막 토큰:", prompt_ids[-1], "(<|assistant|>)")
print("FAST_TOKENIZER_AND_BOUNDARY_OK")


In [ ]:
# forget5 + retain5 평가에 필요한 40개 이미지만 공개 archive에서 복원합니다.
import json
import shutil
import zipfile
from collections import defaultdict
from pathlib import Path

from huggingface_hub import HfApi, hf_hub_download
from PIL import Image

records = [
    json.loads(line)
    for line in (repo / "dataset/full.json").read_text().splitlines()
    if line.strip()
][:400]
split_info = json.loads((repo / "dataset/split.json").read_text())
selected_ids = set(split_info["forget5"] + split_info["retain5"])
needed = {
    row["image_path"]
    for row in records
    if row["unique_id"] in selected_ids
}
assert len(needed) == 40

revision = HfApi().dataset_info("gray311/FIUBench").sha
archive = hf_hub_download(
    repo_id="gray311/FIUBench",
    filename="SFHQ.zip",
    repo_type="dataset",
    revision=revision,
    local_dir="/kaggle/working/fiu_download",
)

with zipfile.ZipFile(archive) as zipped:
    entries = defaultdict(list)
    for member in zipped.infolist():
        path = Path(member.filename)
        if (
            not member.is_dir()
            and "__MACOSX" not in path.parts
            and not path.name.startswith("._")
        ):
            entries[path.name].append(member)

    for image_path in needed:
        destination = (repo / image_path).resolve()
        assert destination.is_relative_to((repo / "dataset").resolve())
        matches = entries[destination.name]
        assert len(matches) == 1, (destination.name, len(matches))
        destination.parent.mkdir(parents=True, exist_ok=True)
        with zipped.open(matches[0]) as source, destination.open("wb") as target:
            shutil.copyfileobj(source, target)
        with Image.open(destination) as image:
            image.verify()

assert all((repo / path).is_file() for path in needed)
(run_dir / "fiu_eval_data_revision.txt").write_text(revision)
print("검증한 이미지:", len(needed))
print("dataset revision:", revision)


In [ ]:
# 원본 파일 수정은 Kaggle T4에서 평가를 실행하는 데 필요한 최소 항목만 적용합니다.
# 평가 흐름, 결과 저장, Truth Ratio 집계는 아래 노트북 셀에서 처리합니다.
import ast
from pathlib import Path

source_path = repo / "evaluate_util.py"
backup = source_path.with_suffix(".py.before_fast_eval")

# 이 세션에서 이전 버전의 과한 패치를 실행했더라도 먼저 원본 백업으로 되돌립니다.
if backup.exists():
    original_source = backup.read_text()
else:
    original_source = source_path.read_text()
source = original_source

# 1) 평가에서 사용하지 않는 함수 import가 forget.py 전체와 DeepSpeed를 끌어오는 문제만 제거합니다.
source = source.replace("from forget import find_all_linear_names\n", "")

# 2) 공개 저장소에 없는 API 모듈 때문에 Core/APE까지 import 단계에서 실패하지 않게 합니다.
# GPT/Gemini 평가는 이 노트북의 평가 범위가 아닙니다.
api_import = """from data_generation.api import (
    GeminiEvaluator, 
    GPTEvaluator, 
    system_message, 
    user_message, 
    jobs
)"""
api_shim = """class GPTEvaluator:
    def __init__(self, *args, **kwargs): pass
    def generate_answer(self, *args, **kwargs):
        raise RuntimeError("GPT-Eval은 이 노트북에 포함되지 않습니다.")
GeminiEvaluator = GPTEvaluator
system_message = user_message = jobs = None
"""
assert api_import in source or api_shim in source
source = source.replace(api_import, api_shim)

# 3) 성공한 평가와 동일한 Fast tokenizer를 명시합니다.
source = source.replace(
    "AutoTokenizer.from_pretrained(cfg.model_path)",
    "AutoTokenizer.from_pretrained(cfg.model_path, use_fast=True)",
)

# 4) T4에서 지원하지 않는 Flash Attention 2만 PyTorch SDPA로 바꿉니다.
source = source.replace(
    'attn_implementation="flash_attention_2"',
    'attn_implementation="sdpa"',
)

# 생성 prompt 경계는 공개 원본 문자열 방식을 그대로 유지합니다.
required_original = """    input_strings = tokenizer.batch_decode(input_ids)

    model_config = get_model_identifiers_from_yaml(cfg.model_family)
    question_start_tag = model_config['question_start_tag']
    answer_tag = model_config['answer_tag']
    answer_tag = answer_tag.replace("\n", "") 

    ground_truth = [s.split(answer_tag)[1].strip(" ") for s in input_strings]
    input_strings = [s.split(answer_tag)[0].strip(" ") for s in input_strings]
    input_strings = [s + answer_tag for s in input_strings]
"""
assert required_original in source, (
    "공개 원본 run_generation 블록이 아닙니다. "
    "저장소를 새로 clone한 뒤 다시 실행하세요."
)
assert 'labels = batch["labels"]\n    input_strings = []' not in source

ast.parse(source)
if not backup.exists():
    backup.write_text(original_source)
source_path.write_text(source)

print("최소 원본 패치 완료")
print("- UNUSED_DEEPSPEED_IMPORT_REMOVED")
print("- UNUSED_API_IMPORT_SHIMMED")
print("- FAST_TOKENIZER=True")
print("- ATTENTION=SDPA")
print("- GENERATION_BOUNDARY=UPSTREAM_STRING_SPLIT")
print("저장/집계 로직은 원본을 수정하지 않고 노트북에서 처리합니다.")


In [ ]:
# 여기부터는 원본 Python 파일을 고치지 않고 노트북에서 평가 단계를 관리합니다.
import yaml

base = yaml.safe_load((repo / "config/eval.yaml").read_text())
base["model_path"] = "gray311/vlm_unlearning_ft_llava_phi_3_mini"
base["LoRA"]["lora_path"] = str(checkpoint)
base["workers"] = 0
base["batch_size"] = 1
base["perturb_batch_size"] = 1
base["overwrite"] = True

# evaluate_util.py 원본은 checkpoint 옆 eval_results에 저장합니다.
# 노트북이 각 단계 종료 직후 Core/APE 전용 폴더로 결과를 복사해 보존합니다.
shared_eval_dir = run_dir / "eval_results"
core_result_dir = run_dir / "core_results"
ape_result_dir = run_dir / "ape_results"
shared_eval_dir.mkdir(parents=True, exist_ok=True)
core_result_dir.mkdir(parents=True, exist_ok=True)
ape_result_dir.mkdir(parents=True, exist_ok=True)

core = dict(base)
core["data_path"] = ["./dataset/full.json"] * 2
core["split_list"] = ["forget5", "retain5"]
core["question_key"] = ["question"] * 2
core["robust_question_key"] = ["question"] * 2
core["answer_key"] = ["answer"] * 2
core["base_answer_key"] = ["paraphrased_answer"] * 2
core["perturbed_answer_key"] = ["perturbed_answer"] * 2
core["eval_task"] = ["eval_forget_log", "eval_retain_log"]
core["robust_eval"] = [["exact_match", "mink"], ["rouge"]]

ape = dict(base)
ape["data_path"] = ["./dataset/full.json"]
ape["split_list"] = ["forget5"]
ape["question_key"] = ["question"]
ape["robust_question_key"] = ["paraphrased_question"]
ape["answer_key"] = ["answer"]
ape["base_answer_key"] = ["paraphrased_answer"]
ape["perturbed_answer_key"] = ["perturbed_answer"]
ape["eval_task"] = ["eval_forget_ape"]
ape["robust_eval"] = [["ape"]]

core_name = f"eval_{METHOD.lower()}_forget5_fast_core"
ape_name = f"eval_{METHOD.lower()}_forget5_fast_ape"
core_path = repo / "config" / f"{core_name}.yaml"
ape_path = repo / "config" / f"{ape_name}.yaml"
core_path.write_text(yaml.safe_dump(core, sort_keys=False, allow_unicode=True))
ape_path.write_text(yaml.safe_dump(ape, sort_keys=False, allow_unicode=True))

print("원본 evaluate_util.py 저장 위치:", shared_eval_dir)
print("Core 보존 위치:", core_result_dir)
print("APE 보존 위치:", ape_result_dir)
print("checkpoint:", checkpoint)


## 1단계: Core 평가와 Truth Ratio — 한 셀로 실행

원본 `evaluate_util.py`가 Core 원본 JSON을 기록하면, 같은 셀의 노트북 코드가 이를 `core_results`로 복사하고 논문 방식 Truth Ratio 및 요약 JSON을 만듭니다. APE를 실행할 필요는 없습니다.


In [ ]:
import os
import shutil
import subprocess

# 실패한 이전 실행의 JSON을 새 결과로 오인하지 않도록 이번 Core 대상만 지웁니다.
shared_forget = shared_eval_dir / "forget5_eval_forget_log.json"
shared_retain = shared_eval_dir / "retain5_eval_retain_log.json"
for stale in [shared_forget, shared_retain]:
    stale.unlink(missing_ok=True)

env = os.environ.copy()
env.update({
    "HF_HOME": "/kaggle/working/hf_cache",
    "TOKENIZERS_PARALLELISM": "false",
    "PYTHONUNBUFFERED": "1",
    "HYDRA_FULL_ERROR": "1",
})
core_log = run_dir / "core_console.log"
command = [sys.executable, "evaluate_util.py", "--config-name", core_name]

with core_log.open("w") as log:
    process = subprocess.Popen(
        command, cwd=repo, env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
        log.write(line)
    return_code = process.wait()

assert return_code == 0, f"Core 평가 실패: exit {return_code}"
assert shared_forget.is_file(), shared_forget
assert shared_retain.is_file(), shared_retain

# Core가 끝난 즉시 노트북이 원본 JSON을 Core 전용 폴더에 보존합니다.
forget_file = core_result_dir / shared_forget.name
retain_file = core_result_dir / shared_retain.name
shutil.copy2(shared_forget, forget_file)
shutil.copy2(shared_retain, retain_file)
shutil.copy2(core_log, core_result_dir / core_log.name)
print("CORE_RAW_JSON_READY")
print(forget_file)
print(retain_file)

# Core가 끝난 직후 원본 JSON을 읽어 Core 요약과 논문 방식 Truth Ratio를 저장합니다.
import json
import math
import statistics

forget_file = core_result_dir / "forget5_eval_forget_log.json"
retain_file = core_result_dir / "retain5_eval_retain_log.json"
forget = json.loads(forget_file.read_text())
retain = json.loads(retain_file.read_text())

def values(obj):
    if isinstance(obj, dict):
        obj = list(obj.values())
    if not isinstance(obj, list):
        return []
    return [float(x) for x in obj if isinstance(x, (int, float))]

def mean_metric(data, key):
    numeric = values(data.get(key))
    return statistics.fmean(numeric) if numeric else None

def paper_truth_scores(data, split):
    paraphrased = data["avg_paraphrased_loss"]
    perturbed = data["average_perturb_loss"]
    common = [key for key in paraphrased if key in perturbed]
    assert common, f"{split}: Truth Ratio 입력 loss가 없습니다."
    scores = []
    for key in common:
        paraphrased_loss = float(paraphrased[key])
        perturb_value = perturbed[key]
        perturb_loss = (
            statistics.fmean(float(x) for x in perturb_value)
            if isinstance(perturb_value, list)
            else float(perturb_value)
        )
        ratio = math.exp(perturb_loss - paraphrased_loss)
        if split == "forget":
            scores.append(min(ratio, 1.0 / ratio))
        else:
            scores.append(max(0.0, 1.0 - 1.0 / ratio))
    return scores

core_summary = {"method": METHOD, "forget": {}, "retain": {}}
print("=" * 68)
print(f"FIUBench {METHOD} Forget5 — FAST tokenizer Core 결과")
print("=" * 68)
for title, key, data in [
    ("Forget", "forget", forget),
    ("Retain", "retain", retain),
]:
    print(f"\n[{title}]")
    for metric in [
        "exact_match", "rouge1_recall", "rougeL_recall",
        "truth_ratio", "mink", "mink++",
    ]:
        value = mean_metric(data, metric)
        if value is not None:
            core_summary[key][metric] = value
            print(f"{metric:20s}: {value:.6f} ({value * 100:.4f}%)")

forget_scores = paper_truth_scores(forget, "forget")
retain_scores = paper_truth_scores(retain, "retain")
forget_truth = statistics.fmean(forget_scores)
retain_truth = statistics.fmean(retain_scores)

truth_summary = {
    "method": METHOD,
    "forget_samples": len(forget_scores),
    "forget_truth_ratio": forget_truth,
    "retain_samples": len(retain_scores),
    "retain_truth_ratio": retain_truth,
    "paper_po_retain_truth_ratio": 0.607,
    "difference_percentage_points": (retain_truth - 0.607) * 100,
}
core_summary["forget_generated_samples"] = len(forget.get("generated_text", {}))
core_summary["retain_generated_samples"] = len(retain.get("generated_text", {}))

core_summary_path = core_result_dir / "core_summary.json"
truth_summary_path = core_result_dir / "paper_truth_ratio_summary.json"
core_summary_path.write_text(json.dumps(core_summary, indent=2, allow_nan=False))
truth_summary_path.write_text(json.dumps(truth_summary, indent=2, allow_nan=False))

print("\n=== 논문 방식 Truth Ratio ===")
print(f"Forget Truth Ratio: {forget_truth * 100:.4f}% ({len(forget_scores)}개)")
print(f"Retain Truth Ratio:  {retain_truth * 100:.4f}% ({len(retain_scores)}개)")
if METHOD == "PO":
    print("논문 PO Retain Truth Ratio: 60.7000%")
    print(f"논문과 차이: {(retain_truth - 0.607) * 100:+.4f}%p")
print("CORE_SUMMARY_JSON_READY:", core_summary_path)
print("TRUTH_RATIO_JSON_READY:", truth_summary_path)


## 2단계: APE 평가 — 선택 실행, 한 셀로 실행

원본 평가가 APE JSON을 기록하면, 같은 셀의 노트북 코드가 이를 `ape_results`로 복사하고 `ape_summary.json`을 만듭니다. Core 결과에는 영향을 주지 않습니다.


In [ ]:
import json
import shutil
import statistics

shared_ape = shared_eval_dir / "forget5_eval_forget_ape.json"
shared_ape.unlink(missing_ok=True)

ape_log = run_dir / "ape_console.log"
command = [sys.executable, "evaluate_util.py", "--config-name", ape_name]

with ape_log.open("w") as log:
    process = subprocess.Popen(
        command, cwd=repo, env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
        log.write(line)
    return_code = process.wait()

assert return_code == 0, f"APE 평가 실패: exit {return_code}"
assert shared_ape.is_file(), shared_ape

# APE가 끝난 즉시 노트북이 원본 JSON을 APE 전용 폴더에 보존합니다.
ape_file = ape_result_dir / shared_ape.name
shutil.copy2(shared_ape, ape_file)
shutil.copy2(ape_log, ape_result_dir / ape_log.name)
print("APE_RAW_JSON_READY:", ape_file)

# APE가 끝난 직후 별도의 요약 JSON을 저장합니다.
ape_file = ape_result_dir / "forget5_eval_forget_ape.json"
ape_result = json.loads(ape_file.read_text())
ape_values = ape_result.get("exact_match", [])
if isinstance(ape_values, dict):
    ape_values = list(ape_values.values())
ape_scores = [
    float(value) for value in ape_values
    if isinstance(value, (int, float))
]
assert len(ape_scores) == 1200, len(ape_scores)
ape_mean = statistics.fmean(ape_scores)
ape_summary = {
    "method": METHOD,
    "samples": len(ape_scores),
    "ape": ape_mean,
    "ape_percent": ape_mean * 100,
}
ape_summary_path = ape_result_dir / "ape_summary.json"
ape_summary_path.write_text(json.dumps(ape_summary, indent=2, allow_nan=False))

print("APE samples:", len(ape_scores))
print("APE mean:", ape_mean)
print("APE percent:", f"{ape_mean * 100:.4f}%")
print("APE_SUMMARY_JSON_READY:", ape_summary_path)


## 보존

- Core 완료 직후: `/kaggle/working/<method>_forget5_fast_eval/core_results`
- APE 완료 직후: `/kaggle/working/<method>_forget5_fast_eval/ape_results`

결과 분리·복사·요약은 `evaluate_util.py`가 아니라 노트북이 담당합니다. Core까지만 완료한 상태에서도 **Save Version → Quick Save**로 Core와 Truth Ratio JSON을 보존할 수 있습니다.
